In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load and clean
coral = pd.read_csv('../data/raw/archive 2/coral.csv')

cols_to_fix = ['Percent_Bleaching', 'Distance_to_Shore', 'Turbidity', 
               'Temperature_Mean', 'SSTA', 'TSA', 'Depth_m']
for col in cols_to_fix:
    coral[col] = pd.to_numeric(coral[col], errors='coerce')

coral_clean = coral.dropna(subset=cols_to_fix).copy()

def bleaching_category(pct):
    if pct == 0:
        return 'None'
    elif pct <= 10:
        return 'Low'
    elif pct <= 50:
        return 'Moderate'
    else:
        return 'Severe'

coral_clean['Bleaching_Category'] = coral_clean['Percent_Bleaching'].apply(bleaching_category)

top_countries = coral_clean['Country_Name'].value_counts().head(15).index
coral_clean['Country_Simplified'] = coral_clean['Country_Name'].apply(
    lambda x: x if x in top_countries else 'Other'
)

categorical_to_encode = ['Ocean_Name', 'Exposure', 'Realm_Name', 'Country_Simplified']
coral_encoded = pd.get_dummies(coral_clean, columns=categorical_to_encode, drop_first=False)

exclude_cols = [
    'Site_ID', 'Sample_ID', 'Data_Source', 'Reef_ID', 'Ecoregion_Name',
    'Country_Name', 'State_Island_Province_Name', 'City_Town_Name', 'Site_Name',
    'Substrate_Name', 'Bleaching_Level', 'Date', 'Site_Comments', 
    'Sample_Comments', 'Bleaching_Comments',
    'Percent_Bleaching', 'Bleaching_Category'
]
feature_cols = [col for col in coral_encoded.columns if col not in exclude_cols]

X = coral_encoded[feature_cols]
y = coral_encoded['Bleaching_Category']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

C:\Users\Asthi\AppData\Local\Temp\ipykernel_8896\2693403701.py:5: DtypeWarning: Columns (0: Distance_to_Shore, 1: Turbidity, 2: Percent_Bleaching) have mixed types. Specify dtype option on import or set low_memory=False.
  coral = pd.read_csv('../data/raw/archive 2/coral.csv')


Train: (22899, 75) Val: (4907, 75) Test: (4908, 75)


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_val_preds = rf_model.predict(X_val)

print("Random Forest - Validation Accuracy:", accuracy_score(y_val, rf_val_preds))
print()
print(classification_report(y_val, rf_val_preds))

ValueError: could not convert string to float: 'nd'

In [3]:
# Check ALL feature columns for non-numeric "nd" values
problem_cols = []
for col in feature_cols:
    if coral_encoded[col].dtype == object:
        problem_cols.append(col)

print("Columns still stored as text/object:", problem_cols)

Columns still stored as text/object: []


In [4]:
# Check actual dtypes more thoroughly (not just object)
print(X_train.dtypes.value_counts())
print()

# Find any column containing the string "nd" anywhere
for col in feature_cols:
    if coral_encoded[col].astype(str).eq('nd').any():
        print(f"Found 'nd' in column: {col}")

bool       32
str        31
float64     9
int64       3
Name: count, dtype: int64

Found 'nd' in column: Percent_Cover
Found 'nd' in column: SSTA_Minimum


In [5]:
# Get all columns currently stored as string type
str_cols = X_train.select_dtypes(include='str').columns.tolist()
print("String-type columns:", str_cols)
print()

# Check each for non-numeric values
for col in str_cols:
    non_numeric = pd.to_numeric(coral_encoded[col], errors='coerce').isna().sum()
    if non_numeric > 0:
        print(f"{col}: {non_numeric} non-numeric values")

String-type columns: ['Percent_Cover', 'ClimSST', 'Temperature_Kelvin', 'Temperature_Minimum', 'Temperature_Maximum', 'Temperature_Kelvin_Standard_Deviation', 'Windspeed', 'SSTA_Standard_Deviation', 'SSTA_Mean', 'SSTA_Minimum', 'SSTA_Maximum', 'SSTA_Frequency', 'SSTA_Frequency_Standard_Deviation', 'SSTA_FrequencyMax', 'SSTA_FrequencyMean', 'SSTA_DHW', 'SSTA_DHW_Standard_Deviation', 'SSTA_DHWMax', 'SSTA_DHWMean', 'TSA_Standard_Deviation', 'TSA_Minimum', 'TSA_Maximum', 'TSA_Mean', 'TSA_Frequency', 'TSA_Frequency_Standard_Deviation', 'TSA_FrequencyMax', 'TSA_FrequencyMean', 'TSA_DHW', 'TSA_DHW_Standard_Deviation', 'TSA_DHWMax', 'TSA_DHWMean']

Percent_Cover: 10153 non-numeric values
SSTA_Minimum: 36 non-numeric values


In [6]:
bad_mask = pd.to_numeric(coral_encoded['Percent_Cover'], errors='coerce').isna()
print(coral_encoded.loc[bad_mask, 'Percent_Cover'].value_counts().head(10))

Percent_Cover
nd    10153
Name: count, dtype: int64


In [7]:
# Convert all string-type feature columns to numeric
str_cols = X_train.select_dtypes(include='str').columns.tolist()

for col in str_cols:
    coral_encoded[col] = pd.to_numeric(coral_encoded[col], errors='coerce')

# Check how much missing data each column now has
missing_counts = coral_encoded[str_cols].isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(missing_counts)

Percent_Cover    10153
SSTA_Minimum        36
dtype: int64


In [8]:
# Drop Percent_Cover column entirely (too much missing data - 31%)
feature_cols_v2 = [col for col in feature_cols if col != 'Percent_Cover']

# Drop rows with missing SSTA_Minimum (only 36 rows, negligible)
coral_final = coral_encoded.dropna(subset=['SSTA_Minimum']).copy()

print("Rows before:", len(coral_encoded))
print("Rows after dropping missing SSTA_Minimum:", len(coral_final))
print("Features before:", len(feature_cols))
print("Features after dropping Percent_Cover:", len(feature_cols_v2))

Rows before: 32714
Rows after dropping missing SSTA_Minimum: 32678
Features before: 75
Features after dropping Percent_Cover: 74


In [9]:
X = coral_final[feature_cols_v2]
y = coral_final['Bleaching_Category']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print()
print("Any remaining missing values in X_train:", X_train.isnull().sum().sum())

Train: (22874, 74) Val: (4902, 74) Test: (4902, 74)

Any remaining missing values in X_train: 0


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_val_preds = rf_model.predict(X_val)

print("Random Forest - Validation Accuracy:", accuracy_score(y_val, rf_val_preds))
print()
print(classification_report(y_val, rf_val_preds))

Random Forest - Validation Accuracy: 0.861485108119135

              precision    recall  f1-score   support

         Low       0.84      0.84      0.84      1558
    Moderate       0.67      0.62      0.64       615
        None       0.94      0.95      0.95      2477
      Severe       0.67      0.65      0.66       252

    accuracy                           0.86      4902
   macro avg       0.78      0.77      0.77      4902
weighted avg       0.86      0.86      0.86      4902



In [11]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

# XGBoost needs numeric labels, not text
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

xgb_model = xgb.XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train_encoded)

xgb_val_preds = xgb_model.predict(X_val)

print("XGBoost - Validation Accuracy:", accuracy_score(y_val_encoded, xgb_val_preds))
print()
print(classification_report(y_val_encoded, xgb_val_preds, target_names=le.classes_))

XGBoost - Validation Accuracy: 0.8472052223582212

              precision    recall  f1-score   support

         Low       0.82      0.80      0.81      1558
    Moderate       0.66      0.61      0.64       615
        None       0.92      0.95      0.93      2477
      Severe       0.71      0.68      0.69       252

    accuracy                           0.85      4902
   macro avg       0.78      0.76      0.77      4902
weighted avg       0.84      0.85      0.85      4902



In [12]:
import numpy as np

importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_cols_v2,
    'importance': importances
}).sort_values('importance', ascending=False)

print(feature_importance_df.head(15))

                    feature  importance
8                   Depth_m    0.074210
1         Longitude_Degrees    0.059655
0          Latitude_Degrees    0.046011
2         Distance_to_Shore    0.042455
7                 Date_Year    0.036271
10       Temperature_Kelvin    0.032305
29                      TSA    0.031689
17  SSTA_Standard_Deviation    0.031247
5                  Date_Day    0.031180
16                     SSTA    0.028959
21           SSTA_Frequency    0.026266
25                 SSTA_DHW    0.026070
9                   ClimSST    0.025852
38                  TSA_DHW    0.023621
6                Date_Month    0.023536


In [13]:
import joblib

# Final evaluation on the untouched test set
rf_test_preds = rf_model.predict(X_test)

print("Random Forest - FINAL Test Accuracy:", accuracy_score(y_test, rf_test_preds))
print()
print(classification_report(y_test, rf_test_preds))

# Save the model
joblib.dump(rf_model, '../models/bleaching_risk_model.pkl')
print("\nModel saved successfully")

Random Forest - FINAL Test Accuracy: 0.8582211342309262

              precision    recall  f1-score   support

         Low       0.83      0.84      0.83      1558
    Moderate       0.69      0.62      0.65       616
        None       0.93      0.95      0.94      2477
      Severe       0.70      0.60      0.65       251

    accuracy                           0.86      4902
   macro avg       0.78      0.75      0.77      4902
weighted avg       0.85      0.86      0.86      4902


Model saved successfully
